<a href="https://colab.research.google.com/github/ibtihal7alharbi-tech/medical-cost-prediction/blob/main/ann_by_banan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K


In [ ]:
df = pd.read_csv("/content/insurance.csv")
df
#Charges col is the target

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [ ]:
df.isnull().sum()

,0
age,0
sex,0
bmi,0
children,0
smoker,0
region,0
charges,0


In [ ]:
df.duplicated().sum()

np.int64(1)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

(1337, 7)

In [ ]:
# Text into numbers
le = LabelEncoder()

df['sex'] = le.fit_transform(df['sex'])       # male/female -> 1/0
df['smoker'] = le.fit_transform(df['smoker']) # yes/no -> 1/0
df['region'] = le.fit_transform(df['region']) # 4 regions -> 0,1,2,3

# drop the target col from X
X = df.drop('charges', axis=1)
y = df['charges']

df

,age,sex,bmi,children,smoker,region,charges
0,19,0,27.900,0,1,3,16884.92400
1,18,1,33.770,1,0,2,1725.55230
2,28,1,33.000,3,0,2,4449.46200
3,33,1,22.705,0,0,1,21984.47061
4,32,1,28.880,0,0,1,3866.85520
...,...,...,...,...,...,...,...
1333,50,1,30.970,3,0,1,10600.54830
1334,18,0,31.920,0,0,0,2205.98080
1335,18,0,36.850,0,0,2,1629.83350
1336,21,0,25.800,0,0,3,2007.94500


In [ ]:
#Datasplitting
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_train = scaler_x.fit_transform(X_train)
X_test = scaler_x.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))


In [ ]:
X_train_fix = np.array(X_train).astype('float32')
y_train_fix = np.array(y_train).astype('float32')
X_test_fix = np.array(X_test).astype('float32')
y_test_fix = np.array(y_test).astype('float32')

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(np.array(y_train).reshape(-1, 1)).astype('float32')
y_test_scaled = scaler_y.transform(np.array(y_test).reshape(-1, 1)).astype('float32')

#Build the model
model = Sequential([
    Dense(128, activation='relu', input_shape=(6,)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1)
])

#Compiling
model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

#Training:
print("Training on Scaled Data...")
model.fit(X_train_fix, y_train_scaled, validation_data=(X_test_fix, y_test_scaled),
          epochs=100, batch_size=32) # verbose=0 if I do not want the epoches to be shown

# 4. Predection + rewrite the original numbers
predictions_scaled = model.predict(X_test_fix)
predictions_final = scaler_y.inverse_transform(predictions_scaled)

# Final Evaluation
final_mae = mean_absolute_error(y_test, predictions_final)
final_r2 = r2_score(y_test, predictions_final)

print(f"MAE: {final_mae:.2f} $")
print(f"  (R2 Score): {final_r2 * 100:.2f}%")

Training on Scaled Data...
Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.5701 - mae: 0.5272 - val_loss: 0.2761 - val_mae: 0.3379
Epoch 2/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.2549 - mae: 0.3435 - val_loss: 0.1827 - val_mae: 0.2522
Epoch 3/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2162 - mae: 0.3073 - val_loss: 0.1590 - val_mae: 0.2648
Epoch 4/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.2043 - mae: 0.2971 - val_loss: 0.1522 - val_mae: 0.2530
Epoch 5/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1946 - mae: 0.2881 - val_loss: 0.1567 - val_mae: 0.2736
Epoch 6/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1771 - mae: 0.2740 - val_loss: 0.1532 - val_mae: 0.2624
Epoch 7/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1878 - mae: 0.2754 - val_loss: 0.1469 - val_mae: 0.2414
Epoch 8/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1807 - mae: 0.2698 - val_loss: 0.1547 - val_mae: 0.2592
Epoch 9/100
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1868 - mae: 0.2

In [ ]:
model.summary()


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_16 (Dense)                │ (None, 128)            │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,653 (108.02 KB)

 Trainable params: 9,217 (36.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 18,436 (72.02 KB)

"In the last period (Epoch 100), the model reached continuous stability, where the error in the training data and the test data were very close (around 0.21 in the adopted format). This persistence proves that the Regularization (such as Dropout and L2) that we used to prevent the models from saving data, became able to predict its knowledge to predict insurance prices for new people with an R2 accuracy of 88%."